# World Cup Predictor: Episode 3a — Poisson GLM
By Karl Estampador

## What this notebook does

Episode 3 fits a **Poisson GLM** to every competitive international match since a
data-selected start year. Two observations per match:

```
log(λ_home) = α[home] + β[away] + γ · I(not neutral)
log(λ_away) = α[away] + β[home]
```

`α` = attack strength, `β` = defensive weakness, `γ` = home advantage.
**No separate intercept** — a single reference-team constraint is sufficient
to identify the two-component model.

## How to use in `predictions.ipynb`

```python
# Before (Episodes 1–2):
%run model.ipynb
%run lr_model.ipynb

# Episode 3:
%run poisson_model.ipynb
```

`predict_group_match`, `predict_knockout_winner`, and `get_elo` are all exposed
so `predictions.ipynb` works without changes. `get_elo` is kept for standings
display and tiebreakers only.

## Function reference

| Function | Returns |
|---|---|
| `predict_group_match(home, away, neutral=True)` | `(home_goals, away_goals, winner_or_None)` |
| `predict_knockout_winner(home, away, neutral=True)` | `(winner, loser, win_prob, lose_prob)` |
| `predict_expected_goals(home, away, neutral=True)` | `(λ_home, λ_away)` |
| `predict_win_probabilities(home, away, neutral=True, group_stage=False)` | `(p_hw, p_draw, p_aw)` |
| `get_elo(team_name)` | `float` (standings / tiebreakers only) |

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import minimize
from scipy.stats import poisson as poisson_dist
from scipy.special import gammaln

warnings.filterwarnings('ignore')
np.random.seed(42)

DATA    = Path('data')
OUTPUTS = Path('outputs')
OUTPUTS.mkdir(exist_ok=True)

print('Imports OK')

Imports OK


---
## Section 1 — Data Preparation

| File | What it contains |
|---|---|
| `data/historical_results/results.csv` | Every international match since 1872 |
| `data/historical_results/shootouts.csv` | Penalty-shootout winners (used to clean draw-rate estimates) |
| `data/teams.csv` | 2026 WC teams and groups |

In [2]:
# --- Load data ---
results_all   = pd.read_csv(DATA / 'historical_results' / 'results.csv',  parse_dates=['date'])
shootouts_all = pd.read_csv(DATA / 'historical_results' / 'shootouts.csv', parse_dates=['date'])
teams_df      = pd.read_csv(DATA / 'teams.csv')

# Guard against schema changes
assert {'date','home_team','away_team','home_score','away_score','tournament','neutral'} \
    <= set(results_all.columns),   'results.csv schema mismatch'
assert {'date','home_team','away_team','winner'} \
    <= set(shootouts_all.columns), 'shootouts.csv schema mismatch'

print(f'results.csv   : {len(results_all):,} rows')
print(f'shootouts.csv : {len(shootouts_all):,} rows')

# --- Tournament filter ---
print('\nAll unique tournament values (first 20 shown):')
all_trn = sorted(results_all['tournament'].unique())
print(all_trn[:20], '...' if len(all_trn) > 20 else '')

friendly_mask  = results_all['tournament'].str.contains('Friendly', case=False, na=False)
df_competitive = results_all[~friendly_mask].copy().reset_index(drop=True)

# Drop rows with missing scores (70 rows in dataset; can't compute log-likelihood on them)
n_before = len(df_competitive)
df_competitive = df_competitive.dropna(subset=['home_score', 'away_score']).copy().reset_index(drop=True)
print(f'Dropped {n_before - len(df_competitive)} rows with missing scores')

print(f'\nAfter removing Friendlies + missing scores: {len(df_competitive):,} rows')
print('Remaining tournament types (first 30):')
print(sorted(df_competitive['tournament'].unique())[:30])

# --- Parse neutral as boolean ---
df_competitive['neutral'] = df_competitive['neutral'].astype(str).str.upper() == 'TRUE'

# --- Decade summary ---
df_competitive['year'] = df_competitive['date'].dt.year
print('\nMatches per decade:')
for decade, cnt in df_competitive.groupby((df_competitive['year'] // 10) * 10).size().items():
    print(f'  {decade}s : {cnt:,}')

# --- Host-nation name check ---
print('\n--- Host-nation name check ---')
print('teams.csv host entries:')
for tn in ['USA', 'Mexico', 'Canada']:
    found = (teams_df['team_name'] == tn).any()
    print(f'  "{tn}" in teams.csv : {found}')

all_results_names = set(results_all['home_team'].tolist() + results_all['away_team'].tolist())
print('results.csv host entries (canonical form):')
for rn in ['United States', 'Mexico', 'Canada']:
    found = rn in all_results_names
    print(f'  "{rn}" in results.csv : {found}')
print('Mapping: teams.csv "USA" → results.csv "United States" (via ELO_ALIASES)')

# --- Team match counts (keyed by raw results.csv names) ---
home_c = df_competitive['home_team'].value_counts()
away_c = df_competitive['away_team'].value_counts()
team_match_counts = home_c.add(away_c, fill_value=0).to_dict()
print(f'\nteam_match_counts : {len(team_match_counts)} teams')
print(f'United States appearances : {int(team_match_counts.get("United States", 0))}')

results.csv   : 49,477 rows
shootouts.csv : 678 rows

All unique tournament values (first 20 shown):
['ABCS Tournament', 'AFC Asian Cup', 'AFC Asian Cup qualification', 'AFC Challenge Cup', 'AFC Challenge Cup qualification', 'AFC Solidarity Cup', 'AFF Championship', 'AFF Championship qualification', 'ASEAN Championship', 'ASEAN Championship qualification', 'African Cup of Nations', 'African Cup of Nations qualification', 'African Friendship Games', 'Afro-Asian Games', 'Al Ain International Cup', 'All-African Games', 'Amílcar Cabral Cup', 'Arab Cup', 'Arab Cup qualification', 'Asian Games'] ...
Dropped 70 rows with missing scores

After removing Friendlies + missing scores: 31,019 rows
Remaining tournament types (first 30):
['ABCS Tournament', 'AFC Asian Cup', 'AFC Asian Cup qualification', 'AFC Challenge Cup', 'AFC Challenge Cup qualification', 'AFC Solidarity Cup', 'AFF Championship', 'AFF Championship qualification', 'ASEAN Championship', 'ASEAN Championship qualification', 'African 

---
## Section 2 — ELO Helpers & Team-Name Canonicalization

**Critical:** `teams.csv` and `results.csv` use different spellings for several 2026
participants (`USA` vs `United States`, `IR Iran` vs `Iran`, etc.). The fitted α/β dicts
are keyed by `results.csv` names. Every public prediction function must call
`_canonical_team()` first so lookups don't silently fail.

`ELO_ALIASES` is copied verbatim from `model.ipynb`/`lr_model.ipynb`. `TEAM_ALIASES`
reuses the same dict — no second copy maintained.

In [3]:
# --- ELO helpers (verbatim from model.ipynb / lr_model.ipynb) ---

elo_raw  = pd.read_csv(DATA / 'elo_ratings_wc2026.csv')
elo_2026 = (
    elo_raw[elo_raw['snapshot_date'] == '2026-05-27']
    .copy()
    .reset_index(drop=True)
)
assert len(elo_2026) > 0, 'No rows for snapshot 2026-05-27'

ELO_ALIASES: dict[str, str] = {
    # teams.csv / results.csv → elo country
    'USA':              'United States',
    'IR Iran':          'Iran',
    "Côte d'Ivoire":    'Ivory Coast',
    'Cabo Verde':       'Cape Verde',
    'DR Congo':         'DR Congo',
    # pass-through variants
    'United States':    'United States',
    'Iran':             'Iran',
    'Ivory Coast':      'Ivory Coast',
    'Cape Verde':       'Cape Verde',
    'Czech Republic':   'Czechia',
    'Czechia':           'Czech Republic',   # reverse: teams.csv 'Czechia' → results.csv 'Czech Republic'
    'Cura\u00e7ao':     'Cura\u00e7ao',
}

_elo_lookup: dict[str, float] = dict(zip(elo_2026['country'], elo_2026['rating']))

def _canonical_elo(name: str) -> str:
    return ELO_ALIASES.get(name, name)

def get_elo(team_name: str) -> float:
    """Pre-tournament ELO rating. Used by predictions.ipynb for standings/tiebreakers only."""
    canon = _canonical_elo(team_name)
    if canon in _elo_lookup:
        return _elo_lookup[canon]
    if team_name in _elo_lookup:
        return _elo_lookup[team_name]
    raise KeyError(f'No Elo rating found for "{team_name}" (canonical: "{canon}")')

print(f'ELO lookup ready ({len(_elo_lookup)} teams)')

# --- Team-name canonicalization ---
TEAM_ALIASES = ELO_ALIASES   # same dict — no second copy

def _canonical_team(name: str) -> str:
    """Translate teams.csv-style names (e.g. 'USA') to results.csv-style names ('United States')."""
    return TEAM_ALIASES.get(name, name)

_HOST_NATIONS = {'United States', 'Mexico', 'Canada'}   # canonical form

def _is_genuine_home(canonical_home: str, neutral: bool = False) -> bool:
    """True only if the home team is a 2026 host nation playing at home."""
    if neutral:
        return False
    return canonical_home in _HOST_NATIONS

# Sanity check
print('\nCanonization sanity checks:')
for src, expected in [('USA', 'United States'), ('IR Iran', 'Iran'), ("Côte d'Ivoire", 'Ivory Coast')]:
    got = _canonical_team(src)
    ok  = '\u2713' if got == expected else '\u2717 FAIL'
    print(f'  _canonical_team("{src}") = "{got}"  {ok}')

ELO lookup ready (48 teams)

Canonization sanity checks:
  _canonical_team("USA") = "United States"  ✓
  _canonical_team("IR Iran") = "Iran"  ✓
  _canonical_team("Côte d'Ivoire") = "Ivory Coast"  ✓


---
## Section 3 — Training Window Selection

Grid-search `start_years = [1990, 1994, 1998, 2002, 2006, 2008]` with flat weights
(no time decay for Poisson). Holdout = the most recent full calendar year.

For each candidate, use a single fixed reference team (alphabetically first in
that window) — **no two-pass procedure here**, since reference choice cannot
change which window wins on RPS. Select the window that **minimises** RPS.

In [4]:
# Shared constants used throughout the notebook
MAX_GOALS = 10

# --- Determine holdout year ---
max_year     = int(df_competitive['year'].max())
current_year = pd.Timestamp.now().year
holdout_year = max_year if max_year < current_year else max_year - 1
print(f'Holdout year       : {holdout_year}')

df_holdout   = df_competitive[df_competitive['year'] == holdout_year].copy()
df_trainable = df_competitive[df_competitive['year'] <  holdout_year].copy()
print(f'Holdout matches    : {len(df_holdout):,}')
print(f'Trainable matches  : {len(df_trainable):,}')


# --- Helper: RPS (lower is better) ---
def _rps(p_home: float, p_draw: float, p_away: float,
         home_goals: int, away_goals: int) -> float:
    """Ranked Probability Score. Ordering: away=0, draw=1, home=2."""
    actual = 2 if home_goals > away_goals else (1 if home_goals == away_goals else 0)
    cum_pred = np.array([p_away, p_away + p_draw, p_away + p_draw + p_home])
    cum_true = np.zeros(3)
    cum_true[actual:] = 1.0
    return float(np.mean((cum_pred - cum_true) ** 2))


# --- Helper: probability matrix ---
def _prob_matrix(lam_home: float, lam_away: float) -> np.ndarray:
    """Joint probability matrix P[i,j] = P(home=i goals, away=j goals)."""
    h = poisson_dist.pmf(np.arange(MAX_GOALS + 1), lam_home)
    a = poisson_dist.pmf(np.arange(MAX_GOALS + 1), lam_away)
    return np.outer(h, a)


def _matrix_outcomes(matrix: np.ndarray) -> tuple[float, float, float]:
    """Extract (p_home_win, p_draw, p_away_win) from joint matrix."""
    p_hw = float(np.sum(np.tril(matrix, -1)))
    p_d  = float(np.sum(np.diag(matrix)))
    p_aw = float(np.sum(np.triu(matrix, 1)))
    return p_hw, p_d, p_aw


# --- Core fit function ---
def _fit_poisson(
    df_train: pd.DataFrame,
    ref_team: str | None = None
) -> tuple[dict, dict, float, float]:
    """
    Fit Poisson GLM via L-BFGS-B.
    Returns (alpha_dict, beta_dict, gamma, negative_log_likelihood).
    Reference team's alpha is pinned to 0 (excluded from param vector).
    """
    teams   = sorted(set(df_train['home_team'].tolist() + df_train['away_team'].tolist()))
    n       = len(teams)
    t2i     = {t: i for i, t in enumerate(teams)}

    if ref_team is None or ref_team not in t2i:
        ref_team = teams[0]
    ref_idx = t2i[ref_team]

    home_idx  = np.array([t2i[t] for t in df_train['home_team']])
    away_idx  = np.array([t2i[t] for t in df_train['away_team']])
    hg        = df_train['home_score'].values.astype(float)
    ag        = df_train['away_score'].values.astype(float)
    neutral   = df_train['neutral'].values.astype(bool)

    # param layout: [alpha_raw (n-1), beta (n), gamma (1)]  total = 2n
    x0        = np.zeros(2 * n)
    x0[-1]    = 0.2
    bounds    = ([(-3.0, 3.0)] * (n - 1) +
                 [(-3.0, 3.0)] * n +
                 [(-1.0, 2.0)])

    def neg_ll(params: np.ndarray) -> float:
        a_raw  = params[:n - 1]
        beta   = params[n - 1: 2 * n - 1]
        gamma  = params[2 * n - 1]
        alpha  = np.insert(a_raw, ref_idx, 0.0)
        h_adv  = np.where(neutral, 0.0, gamma)
        loglh  = alpha[home_idx] + beta[away_idx] + h_adv
        logla  = alpha[away_idx] + beta[home_idx]
        lh, la = np.exp(loglh), np.exp(logla)
        # np.where guards against 0 * -inf = nan when lambda is tiny
        ll = (np.where(hg > 0, hg * loglh, 0.0) - lh - gammaln(hg + 1) +
              np.where(ag > 0, ag * logla, 0.0) - la - gammaln(ag + 1))
        result = ll.sum()
        return -result if np.isfinite(result) else 1e10

    res     = minimize(neg_ll, x0, method='L-BFGS-B', bounds=bounds)
    a_raw   = res.x[:n - 1]
    beta    = res.x[n - 1: 2 * n - 1]
    gamma   = float(res.x[2 * n - 1])
    alpha   = np.insert(a_raw, ref_idx, 0.0)
    nll_val = float(neg_ll(res.x))   # re-evaluate at final params; avoids nan in res.fun
    return dict(zip(teams, alpha)), dict(zip(teams, beta)), gamma, nll_val


# --- Helper: holdout RPS ---
def _holdout_rps(
    alpha_d: dict, beta_d: dict, gamma_val: float,
    df_eval: pd.DataFrame
) -> float:
    """Mean holdout RPS. Teams absent from training use mean fallback."""
    a_fb = float(np.mean(list(alpha_d.values()))) if alpha_d else 0.0
    b_fb = float(np.mean(list(beta_d.values())))  if beta_d  else 0.0
    scores = []
    for _, r in df_eval.iterrows():
        ah = alpha_d.get(r['home_team'], a_fb)
        bh = beta_d.get(r['home_team'],  b_fb)
        aa = alpha_d.get(r['away_team'], a_fb)
        ba = beta_d.get(r['away_team'],  b_fb)
        g  = 0.0 if r['neutral'] else gamma_val
        lh = np.exp(ah + ba + g)
        la = np.exp(aa + bh)
        pw, pd_, pa = _matrix_outcomes(_prob_matrix(lh, la))
        scores.append(_rps(pw, pd_, pa, r['home_score'], r['away_score']))
    return float(np.mean(scores)) if scores else float('nan')

print('Helper functions defined.')

Holdout year       : 2025
Holdout matches    : 781
Trainable matches  : 30,133
Helper functions defined.


In [5]:
start_years  = [1990, 1994, 1998, 2002, 2006, 2008]
grid_results = []

print(f'Grid search — holdout year: {holdout_year}')
print(f'{"start_year":>12}  {"train_rows":>10}  {"holdout_rps":>12}')
print('-' * 40)

for sy in start_years:
    df_tr  = df_trainable[df_trainable['year'] >= sy].copy()
    teams_ = sorted(set(df_tr['home_team'].tolist() + df_tr['away_team'].tolist()))
    ref    = teams_[0]   # fixed arbitrary reference — no two-pass needed here
    ad, bd, gv, _ = _fit_poisson(df_tr, ref_team=ref)
    rps    = _holdout_rps(ad, bd, gv, df_holdout)
    grid_results.append({'start_year': sy, 'train_rows': len(df_tr), 'holdout_rps': rps})
    print(f'{sy:>12}  {len(df_tr):>10,}  {rps:>12.5f}')

best_sy = min(grid_results, key=lambda x: x['holdout_rps'])['start_year']
print(f'\nSelected start_year : {best_sy}  (minimises holdout RPS)')
BEST_START_YEAR = best_sy

Grid search — holdout year: 2025
  start_year  train_rows   holdout_rps
----------------------------------------


        1990      20,574       0.11042


        1994      18,999       0.11076


        1998      16,969       0.10979


        2002      14,576       0.10888


        2006      12,305       0.10747


        2008      11,157       0.10753

Selected start_year : 2006  (minimises holdout RPS)


---
## Section 4 — Final Model Fit

Two-pass reference-team selection (run once, on the final training window):
1. Preliminary fit pinning the alphabetically-first team to α=0 (purely to rank teams)
2. Pick the team closest to the median α
3. Refit using that team as the permanent reference

In [6]:
df_final_train = df_competitive[df_competitive['year'] >= BEST_START_YEAR].copy()
print(f'Final training window : {BEST_START_YEAR}–present  ({len(df_final_train):,} rows)')

# Pass 1 — rank all teams' alpha
teams_final = sorted(set(df_final_train['home_team'].tolist() + df_final_train['away_team'].tolist()))
ref_p1      = teams_final[0]
print(f'\nPass 1: reference = "{ref_p1}"')
alpha_p1, _, _, _ = _fit_poisson(df_final_train, ref_team=ref_p1)

# Pass 2 — pick median-alpha team as reference
median_alpha = float(np.median(list(alpha_p1.values())))
ref_final    = min(alpha_p1.keys(), key=lambda t: abs(alpha_p1[t] - median_alpha))
print(f'Median α = {median_alpha:.4f}  →  reference = "{ref_final}" (α={alpha_p1[ref_final]:.4f})')

alpha_dict, beta_dict, GAMMA, nll_final = _fit_poisson(df_final_train, ref_team=ref_final)
REFERENCE_TEAM = ref_final

assert np.isfinite(nll_final), f'Final NLL is not finite: {nll_final}'
print(f'\nFinal fit NLL  : {nll_final:.2f}')
print(f'γ (home adv)   : {GAMMA:.4f}')
print(f'Reference team : "{REFERENCE_TEAM}"')

alpha_s = pd.Series(alpha_dict).sort_values(ascending=False)
print('\nTop 5 attack (α):')
for t, v in alpha_s.head(5).items():
    print(f'  {t:<32} {v:+.4f}')
print('Bottom 5 attack (α):')
for t, v in alpha_s.tail(5).items():
    print(f'  {t:<32} {v:+.4f}')

Final training window : 2006–present  (13,191 rows)

Pass 1: reference = "Abkhazia"


Median α = 0.1314  →  reference = "Finland" (α=0.1319)



Final fit NLL  : 37006.80
γ (home adv)   : 0.2735
Reference team : "Finland"

Top 5 attack (α):
  Brazil                           +1.2950
  Quebec                           +1.1884
  Argentina                        +1.1604
  Germany                          +1.0582
  Mexico                           +1.0469
Bottom 5 attack (α):
  American Samoa                   -1.3017
  Liechtenstein                    -1.3808
  Anguilla                         -1.4075
  Andorra                          -1.4414
  San Marino                       -2.1151


---
## Section 5 — Group Stage Draw Inflation

Poisson systematically under-predicts draws. We measure the residual gap
against historical FIFA World Cup group-stage matches and correct it.

`results.csv` has no round/stage column, so we filter on `tournament ==
'FIFA World Cup'` then **exclude penalty-shootout-decided matches** (joined
from `shootouts.csv` on date + teams) to remove the worst source of
false 'draws' from knockout rounds.

Draw inflation is clipped to [1.0, 1.5] — based on long-run historical
calibration, not on what's happened in WC2026 so far.

In [7]:
WC_STR = 'FIFA World Cup'
df_wc  = df_competitive[df_competitive['tournament'] == WC_STR].copy()

# Exclude penalty-shootout matches
shootout_keys = set(
    zip(
        shootouts_all['date'].dt.strftime('%Y-%m-%d'),
        shootouts_all['home_team'],
        shootouts_all['away_team']
    )
)
df_wc['_key'] = list(zip(
    df_wc['date'].dt.strftime('%Y-%m-%d'),
    df_wc['home_team'],
    df_wc['away_team']
))
df_wc_ns = df_wc[~df_wc['_key'].isin(shootout_keys)].copy()

empirical_draw_rate = float((df_wc_ns['home_score'] == df_wc_ns['away_score']).mean())
print(f'WC matches (excl. shootouts) : {len(df_wc_ns)}')
print(f'Empirical draw rate          : {empirical_draw_rate:.4f}  ({empirical_draw_rate*100:.1f}%)')
if not 0.15 <= empirical_draw_rate <= 0.40:
    print('  WARNING: draw rate outside expected 15–40% range — check filtering')

# Model average draw on the same matches
a_fb = float(np.mean(list(alpha_dict.values())))
b_fb = float(np.mean(list(beta_dict.values())))
model_draws = []
for _, row in df_wc_ns.iterrows():
    ah = alpha_dict.get(row['home_team'], a_fb)
    bh = beta_dict.get( row['home_team'], b_fb)
    aa = alpha_dict.get(row['away_team'], a_fb)
    ba = beta_dict.get( row['away_team'], b_fb)
    g  = 0.0 if row['neutral'] else GAMMA
    lh = np.exp(ah + ba + g)
    la = np.exp(aa + bh)
    _, pd_, _ = _matrix_outcomes(_prob_matrix(lh, la))
    model_draws.append(pd_)

MODEL_AVG_DRAW  = float(np.mean(model_draws))
DRAW_INFLATION  = float(np.clip(empirical_draw_rate / MODEL_AVG_DRAW, 1.0, 1.5))

print(f'Model avg draw probability   : {MODEL_AVG_DRAW:.4f}')
print(f'Draw inflation factor        : {DRAW_INFLATION:.4f}  (clipped to [1.0, 1.5])')

WC matches (excl. shootouts) : 931
Empirical draw rate          : 0.1923  (19.2%)


Model avg draw probability   : 0.2444
Draw inflation factor        : 1.0000  (clipped to [1.0, 1.5])


---
## Section 6 — Shrinkage Fallback

Teams with fewer than 5 training matches have their α/β blended toward the
population mean. Unknown teams use the mean directly.

**Verification note:** if `"USA"` or `"United States"` appears in the warning
list below, canonicalization is broken — that team has hundreds of training
matches and should never need shrinkage.

In [8]:
alpha_fallback = float(np.mean(list(alpha_dict.values())))
beta_fallback  = float(np.mean(list(beta_dict.values())))
print(f'α fallback (mean) : {alpha_fallback:.4f}')
print(f'β fallback (mean) : {beta_fallback:.4f}')

print('\n--- Fallback / shrinkage for 2026 WC teams ---')
print(f'  {"Team (teams.csv)":<30} {"Canonical":<30} {"N matches":>10}  Status')
print('  ' + '-' * 80)
any_issues = False
for _, row in teams_df.sort_values('team_name').iterrows():
    tn    = row['team_name']
    canon = _canonical_team(tn)
    n     = int(team_match_counts.get(canon, 0))
    if n == 0:
        status = 'FALLBACK (no training data)'
        any_issues = True
    elif n < 5:
        status = f'SHRINKAGE (n={n})'
        any_issues = True
    else:
        continue
    print(f'  {tn:<30} {canon:<30} {n:>10}  {status}')

if not any_issues:
    print('  All 2026 WC teams have sufficient training data (n >= 5).')


def _team_params(canonical: str) -> tuple[float, float]:
    """Return (alpha, beta) for a team, applying Bayesian shrinkage if n < 5."""
    n = int(team_match_counts.get(canonical, 0))
    if canonical not in alpha_dict:
        return alpha_fallback, beta_fallback
    a, b = alpha_dict[canonical], beta_dict[canonical]
    if n < 5:
        w = n / (n + 5)
        return (w * a + (1 - w) * alpha_fallback,
                w * b + (1 - w) * beta_fallback)
    return a, b

α fallback (mean) : 0.0793
β fallback (mean) : 0.0917

--- Fallback / shrinkage for 2026 WC teams ---
  Team (teams.csv)               Canonical                       N matches  Status
  --------------------------------------------------------------------------------
  All 2026 WC teams have sufficient training data (n >= 5).


---
## Section 7 — Prediction Functions

All four public functions canonicalize `home`/`away` via `_canonical_team()`
as their first action, then use canonical names for every downstream lookup.

In [9]:
def _inflate_draws(matrix: np.ndarray) -> np.ndarray:
    """Apply DRAW_INFLATION to the diagonal and renormalize."""
    m  = matrix.copy()
    di = np.arange(m.shape[0])
    m[di, di] *= DRAW_INFLATION
    m /= m.sum()
    return m


def predict_expected_goals(
    home: str, away: str, neutral: bool = True
) -> tuple[float, float]:
    """Return (lambda_home, lambda_away) from fitted Poisson parameters."""
    ch, ca   = _canonical_team(home), _canonical_team(away)
    a_h, b_h = _team_params(ch)
    a_a, b_a = _team_params(ca)
    g_eff    = GAMMA if _is_genuine_home(ch, neutral) else 0.0
    return float(np.exp(a_h + b_a + g_eff)), float(np.exp(a_a + b_h))


def predict_win_probabilities(
    home: str, away: str,
    neutral: bool = True, group_stage: bool = False
) -> tuple[float, float, float]:
    """Return (p_home_win, p_draw, p_away_win)."""
    lh, la = predict_expected_goals(home, away, neutral)
    matrix = _prob_matrix(lh, la)
    if group_stage:
        matrix = _inflate_draws(matrix)
    return _matrix_outcomes(matrix)


def predict_group_match(
    home: str, away: str, neutral: bool = True
) -> tuple[int, int, str | None]:
    """Group-stage prediction (draws allowed). Draw inflation always applied."""
    lh, la = predict_expected_goals(home, away, neutral)
    matrix = _inflate_draws(_prob_matrix(lh, la))
    ri, ci = np.unravel_index(np.argmax(matrix), matrix.shape)
    ri, ci = int(ri), int(ci)
    if ri == ci:
        return ri, ci, None
    return ri, ci, (home if ri > ci else away)


def predict_knockout_winner(
    home: str, away: str, neutral: bool = True
) -> tuple[str, str, float, float]:
    """Knockout prediction (no draws). Renormalize between home/away only."""
    p_hw, _, p_aw = predict_win_probabilities(home, away, neutral)
    total = p_hw + p_aw
    if total <= 0:
        total = 1.0
    p_h, p_a = p_hw / total, p_aw / total
    if p_h >= p_a:
        return home, away, p_h, p_a
    return away, home, p_a, p_h


print('All prediction functions defined.')

All prediction functions defined.


---
## Section 8 — Integration Tests

**Verification:** check the Section 6 fallback log first. If `"USA"` or
`"United States"` appears there, canonicalization is broken and the fourth
row below will be wrong.

The fourth match (`USA vs Mexico, neutral=False`) is the critical one — it's
the only case that exercises the `_is_genuine_home` / home-advantage code path.
The first three are all neutral under WC2026 hosting.

In [10]:
test_matches = [
    ('France',   'Argentina', True),
    ('Spain',    'Brazil',    True),
    ('Germany',  'England',   True),
    ('USA',      'Mexico',    False),  # host nation home — exercises canonicalization + γ path
]

print('=' * 80)
print('INTEGRATION TESTS — Poisson GLM')
print('=' * 80)

# Expected goals & win probabilities
print('\n--- Expected Goals & Win Probabilities ---')
hdr = f'{"Match":<30} {"neut":>5}  {"λ_home":>7} {"λ_away":>7}  {"P(hw)":>7} {"P(draw)":>7} {"P(aw)":>7}'
print(hdr)
print('-' * len(hdr))
for home, away, neutral in test_matches:
    lh, la     = predict_expected_goals(home, away, neutral)
    pw, pd_, pa = predict_win_probabilities(home, away, neutral)
    label      = f'{home} vs {away}'
    print(f'{label:<30} {str(neutral):>5}  {lh:>7.3f} {la:>7.3f}  {pw:>7.3f} {pd_:>7.3f} {pa:>7.3f}')

# Group match
print('\n--- Group match (draw inflation applied) ---')
print(f'{"Match":<30} {"neut":>5}  {"HG":>3} {"AG":>3}  Result')
print('-' * 55)
for home, away, neutral in test_matches:
    hg, ag, winner = predict_group_match(home, away, neutral)
    label  = f'{home} vs {away}'
    result = winner if winner is not None else 'Draw'
    print(f'{label:<30} {str(neutral):>5}  {hg:>3} {ag:>3}  {result}')

# Knockout
print('\n--- Knockout (no draws, renormalized) ---')
print(f'{"Match":<30} {"neut":>5}  {"Winner":<22} {"Win prob":>9}')
print('-' * 70)
for home, away, neutral in test_matches:
    w, l, wp, lp = predict_knockout_winner(home, away, neutral)
    label = f'{home} vs {away}'
    print(f'{label:<30} {str(neutral):>5}  {w:<22} {wp:>9.4f}')

INTEGRATION TESTS — Poisson GLM

--- Expected Goals & Win Probabilities ---
Match                           neut   λ_home  λ_away    P(hw) P(draw)   P(aw)
------------------------------------------------------------------------------
France vs Argentina             True    0.663   1.387    0.178   0.277   0.544
Spain vs Brazil                 True    0.788   1.480    0.200   0.264   0.536
Germany vs England              True    1.203   1.178    0.367   0.278   0.355
USA vs Mexico                  False    1.329   1.317    0.372   0.261   0.367

--- Group match (draw inflation applied) ---
Match                           neut   HG  AG  Result
-------------------------------------------------------
France vs Argentina             True    0   1  Argentina
Spain vs Brazil                 True    0   1  Brazil
Germany vs England              True    1   1  Draw
USA vs Mexico                  False    1   1  Draw

--- Knockout (no draws, renormalized) ---
Match                           neut

In [11]:
print('=' * 60)
print('MODEL READY — Poisson GLM (Episode 3)')
print('=' * 60)
print(f'  Selected start year  : {BEST_START_YEAR}')
print(f'  Reference team       : "{REFERENCE_TEAM}"')
print(f'  γ (home advantage)   : {GAMMA:.4f}')
print(f'  Draw inflation       : {DRAW_INFLATION:.4f}')
print(f'  α fallback           : {alpha_fallback:.4f}')
print(f'  β fallback           : {beta_fallback:.4f}')
print()
print('To use in predictions.ipynb:')
print('  %run poisson_model.ipynb')

MODEL READY — Poisson GLM (Episode 3)
  Selected start year  : 2006
  Reference team       : "Finland"
  γ (home advantage)   : 0.2735
  Draw inflation       : 1.0000
  α fallback           : 0.0793
  β fallback           : 0.0917

To use in predictions.ipynb:
  %run poisson_model.ipynb
